# Elo TC1 Manual Baseline Notes

This notebook is an evidence artifact for the `tc1_from_scratch` human baseline.
It records dataset inspection and plain-language choices only.
The final bank-id mapping lives in `manual_selection_worksheet.md`, not in this notebook.


In [ ]:
import pandas as pd

REF_DATE = pd.Timestamp("2018-02-01")
train = pd.read_csv("../input/train.csv", parse_dates=["first_active_month"])
test = pd.read_csv("../input/test.csv", parse_dates=["first_active_month"])

print("train shape", train.shape)
print("test shape", test.shape)
print(train[["first_active_month", "feature_1", "feature_2", "feature_3"]].head())
print(train[["feature_1", "feature_2", "feature_3"]].nunique())


## Card Age And Base Card Features

First-pass reasoning:

- `first_active_month` is the cleanest card-age signal in the base table
- the three anonymized base-card features are low-cardinality and naturally invite simple dummy expansion
- keep the base-card block without turning the notebook into a mapping guide


In [ ]:
for frame in (train, test):
    frame["year"] = frame["first_active_month"].dt.year
    frame["month"] = frame["first_active_month"].dt.month
    frame["elapsed_time"] = (REF_DATE - frame["first_active_month"]).dt.days

print(train[["first_active_month", "year", "month", "elapsed_time"]].head())
print(pd.get_dummies(train[["feature_1", "feature_2", "feature_3"]].head(), columns=["feature_1", "feature_2", "feature_3"]).head())


## Transaction Tables, Recency, And Aggregation Direction

Second-pass reasoning:

- the transaction tables are where the competition gets most of its signal
- `authorized_flag` and `category_1` are simple Y/N fields that are easy to binarize without much controversy
- `month_diff` is the clearest repeated recency feature because it combines purchase timing with `month_lag`
- keep the numeric card-level aggregation path on both transaction tables, but stop before the full category-dummy plus categorical-aggregation branch


In [ ]:
historical = pd.read_csv("../input/historical_transactions.csv", parse_dates=["purchase_date"])
new = pd.read_csv("../input/new_merchant_transactions.csv", parse_dates=["purchase_date"])

for name, frame in (("historical", historical), ("new", new)):
    preview = frame[["authorized_flag", "category_1", "category_2", "category_3", "purchase_amount", "installments", "month_lag"]].head()
    print(name, frame.shape)
    print(preview)
    frame = frame.copy()
    frame["authorized_flag_encoded"] = frame["authorized_flag"].map({"Y": 1, "N": 0})
    frame["category_1_encoded"] = frame["category_1"].map({"Y": 1, "N": 0})
    frame["month_diff"] = ((REF_DATE - frame["purchase_date"]).dt.days // 30) + frame["month_lag"]
    numeric_preview = frame.groupby("card_id")[["purchase_amount", "installments", "month_lag", "month_diff"]].agg(["sum", "mean", "max", "min", "std"]).head()
    print(name, "numeric aggregate preview")
    print(numeric_preview)


## Manual Takeaways From Inspection

- keep the card-age features from `first_active_month`
- keep the simple dummy expansion for the three base-card features
- binarize the two obvious Y/N transaction flags
- keep `month_diff` as the main recency feature in the transaction tables
- aggregate the strongest numeric transaction signals back to `card_id` from both historical and new-merchant tables
- stop before the heavier `category_2` / `category_3` dummy expansion and categorical aggregation branch in this first-pass baseline
